# ADTC 2026 — QLoRA fine-tune (Qwen2.5-0.5B, agriculture)

Runs the whole training pipeline on a **free Colab T4**, top to bottom:
clone repo → install → prepare data → QLoRA fine-tune → merge → export **Q4_K_M GGUF** → download.

**Before you start:** Runtime → Change runtime type → **T4 GPU** → Save. Then run each cell top to bottom.

The job is tiny (0.5B model, 280 examples, 3 epochs) — expect **a few minutes** on the T4.

> If a cell errors, don't debug alone — copy the full traceback back to Claude Code and it'll patch the script. First real GPU run is the integration test; one or two dependency fixes are normal.

## 1. Confirm the GPU
Should print a **Tesla T4**. If it says 'command not found' or no GPU, fix the runtime type above first.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

## 2. Clone your repo
Public repo — no token needed. Re-runnable (pulls latest if already cloned).

In [ ]:
import os
REPO = '/content/adtc-2026-agriculture'
if os.path.isdir(REPO):
    !cd {REPO} && git pull --ff-only
else:
    !git clone https://github.com/SKI-LEH/adtc-2026-agriculture.git {REPO}
%cd {REPO}/training
!ls -la

## 3. Install training deps
Colab already ships a CUDA-matched **torch** — we do NOT reinstall it (that's the usual source of breakage). Only the extras.

In [ ]:
!pip install -q -U "transformers>=4.46" "peft>=0.13" "trl>=0.13,<0.16" "bitsandbytes>=0.44" "accelerate>=1.0" "datasets>=3.0"
import torch, transformers, trl, peft
print('torch', torch.__version__, '| cuda', torch.cuda.is_available(), '| bf16?', torch.cuda.is_bf16_supported())
print('transformers', transformers.__version__, '| trl', trl.__version__, '| peft', peft.__version__)

## 4. Build the train/eval splits
Stdlib-only, deterministic (seeded). Prints the language × format distribution so you can eyeball it.

In [ ]:
!python prepare_data.py

## 5. QLoRA fine-tune
4-bit NF4 load + LoRA adapter. The script auto-detects that the T4 can't do bf16 and uses **fp16** — you'll see `compute dtype: fp16` in the first line of output. Writes `output/adapter/`.

In [ ]:
!python train_qlora.py

## 6. Merge adapter → fp16, then export Q4_K_M GGUF
Clones llama.cpp only for its `convert_hf_to_gguf.py`, builds `llama-quantize` from source (Colab has no prebuilt Windows exe), then runs both export steps. Writes `output/gguf/agri-qwen2.5-0.5b-Q4_K_M.gguf`.

In [ ]:
# 6a. merge LoRA into fp16 HF weights
!python merge_and_export.py --merge

In [ ]:
# 6b. get llama.cpp (converter + quantizer). Build just the quantize tool.
%cd /content
if not os.path.isdir('/content/llama.cpp'):
    !git clone --depth 1 https://github.com/ggerganov/llama.cpp
!pip install -q gguf sentencepiece protobuf
!cd /content/llama.cpp && cmake -B build -DLLAMA_CURL=OFF > /tmp/cmake.log 2>&1 && cmake --build build --config Release -j --target llama-quantize > /tmp/build.log 2>&1 && echo 'built llama-quantize' || (echo 'BUILD FAILED — tail of log:'; tail -20 /tmp/build.log)
# expose the freshly-built binary where merge_and_export.py looks (PATH fallback)
import shutil, glob
qbin = glob.glob('/content/llama.cpp/build/bin/llama-quantize') + glob.glob('/content/llama.cpp/build/bin/Release/llama-quantize')
if qbin:
    shutil.copy(qbin[0], '/usr/local/bin/llama-quantize')
    print('llama-quantize ready at /usr/local/bin/llama-quantize')
else:
    print('WARNING: llama-quantize binary not found — check /tmp/build.log')

In [ ]:
# 6c. convert merged model → fp16 GGUF → quantize to Q4_K_M
%cd /content/adtc-2026-agriculture/training
!python merge_and_export.py --gguf --llama-cpp /content/llama.cpp

## 7. Verify + download the GGUF
Confirms the file exists and its size (~380 MB expected), then downloads it to your machine. This is the weight file you'll host publicly and point `download_model.sh` at.

In [ ]:
GGUF = '/content/adtc-2026-agriculture/training/output/gguf/agri-qwen2.5-0.5b-Q4_K_M.gguf'
import os
assert os.path.exists(GGUF), f'not found: {GGUF} — check step 6 output'
print(f'{GGUF}\n{os.path.getsize(GGUF)/1e6:.1f} MB')
from google.colab import files
files.download(GGUF)

## Done — next steps (back in Claude Code)
1. **Score it**: run this GGUF through `adtc-profiler` in participant mode; confirm `params_match=true`, peak RSS < 7 GB, TPS vs the 0.5B baseline.
2. **Host it**: upload to a public Hugging Face repo.
3. **Build the submission**: fork the ADTC template, fill `metadata.json` / `download_model.sh` / `REPORT.md`, verify end-to-end, submit the repo URL on Devpost.

Claude Code can draft steps 2–3 for you now.